# NLP Lab 01: Introduction to Natural Language Processing

## Setting Up NLP Libraries

We will use three libraries throughout this lab:

| Library | Best for | Language focus |
|---|---|---|
| **NLTK** (Natural Language Toolkit) | Teaching/classic NLP: tokenization, POS tagging, simple parsing, corpora | Mainly English |
| **spaCy** | Fast, production-grade pipelines: tokenization, POS, dependency parsing, NER | Many languages (via models), mainly English here |
| **PyThaiNLP** | Thai-specific NLP: word segmentation, POS tagging, NER, stopwords, transliteration | Thai |

Run the cell below once to install/download everything you need. If you're on a shared or offline machine, some downloads (like the spaCy model) may need to be run separately or may already be cached.

In [ ]:
# Run this once. Uncomment if the packages are not already installed in your environment.
# %pip install -q nltk spacy pythainlp scikit-learn
# %python -m spacy download en_core_web_sm
print("If needed, uncomment the install lines above and re-run this cell.")

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ------ --------------------------------- 2.1/12.8 MB 13.0 MB/s eta 0:00:01
     ----------------------- ---------------- 7.6/12.8 MB 20.4 MB/s eta 0:00:01
     ------------------------------- ------- 10.5/12.8 MB 21.1 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 17.1 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
If needed, uncomment the install lines above and re-run this cell.


In [7]:
import nltk

# # Download the small set of NLTK data packages we need for this lab.
for pkg in ["punkt", "punkt_tab", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng",
            "stopwords", "maxent_ne_chunker", "maxent_ne_chunker_tab", "words"]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print(f"Skipped {pkg}: {e}")

print("NLTK data ready.")

NLTK data ready.


## NLTK example (English)

NLTK gives us the classic NLP pipeline building blocks: tokenization, POS tagging, and a simple chunk-based NER.
- https://www.nltk.org//
- https://www.geeksforgeeks.org/nlp/nltk-tutorial/ 


In [13]:
from nltk.tokenize import word_tokenize
from nltk import pos_tag, ne_chunk
from nltk.corpus import stopwords

sentence = "Nvidia Corporation is an American multinational technology company incorporated in Delaware and based in Santa Clara, California. It designs graphics processing units (GPUs) for the gaming and professional markets, as well as system on a chip units (SoCs) for the mobile computing and automotive market. Its primary GPU product line, labeled "

# 1. Tokenization: split raw text into words/punctuation tokens
tokens = word_tokenize(sentence)
print("Tokens:", tokens)

# 2. Part-of-speech (POS) tagging: label each token's grammatical role
tagged = pos_tag(tokens)
print("\nPOS tags:", tagged)

# 3. Named Entity Recognition (a simple, rule/statistics-based chunker)
tree = ne_chunk(tagged)
print("\nNamed Entity Tree:", tree)
print("\nNamed entities:")
for subtree in tree:
    if hasattr(subtree, "label"):
        entity_text = " ".join(word for word, tag in subtree.leaves())
        print(f"  {entity_text}  ->  {subtree.label()}")

# 4. Stopword removal (common, low-information words)
stop_words = set(stopwords.words("english"))
filtered = [w for w in tokens if w.lower() not in stop_words and w.isalpha()]
print("\nTokens without stopwords:", filtered)

Tokens: ['Nvidia', 'Corporation', 'is', 'an', 'American', 'multinational', 'technology', 'company', 'incorporated', 'in', 'Delaware', 'and', 'based', 'in', 'Santa', 'Clara', ',', 'California', '.', 'It', 'designs', 'graphics', 'processing', 'units', '(', 'GPUs', ')', 'for', 'the', 'gaming', 'and', 'professional', 'markets', ',', 'as', 'well', 'as', 'system', 'on', 'a', 'chip', 'units', '(', 'SoCs', ')', 'for', 'the', 'mobile', 'computing', 'and', 'automotive', 'market', '.', 'Its', 'primary', 'GPU', 'product', 'line', ',', 'labeled']

POS tags: [('Nvidia', 'NNP'), ('Corporation', 'NNP'), ('is', 'VBZ'), ('an', 'DT'), ('American', 'JJ'), ('multinational', 'NN'), ('technology', 'NN'), ('company', 'NN'), ('incorporated', 'VBN'), ('in', 'IN'), ('Delaware', 'NNP'), ('and', 'CC'), ('based', 'VBN'), ('in', 'IN'), ('Santa', 'NNP'), ('Clara', 'NNP'), (',', ','), ('California', 'NNP'), ('.', '.'), ('It', 'PRP'), ('designs', 'VBZ'), ('graphics', 'NNS'), ('processing', 'VBG'), ('units', 'NNS'), ('(

## spaCy example (English)

spaCy bundles tokenization, POS tagging, dependency parsing, and NER into a single, fast pipeline (`nlp(text)`).
- https://spacy.io/
- https://www.tutorialspoint.com/spacy/index.htm

In [53]:
import spacy

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    raise RuntimeError(
        "Model not found. Run: python -m spacy download en_core_web_sm"
    )

doc = nlp("Nvidia Corporation is an American multinational technology company incorporated in Delaware and based in Santa Clara, California. It designs graphics processing units (GPUs) for the gaming and professional markets, as well as system on a chip units (SoCs) for the mobile computing and automotive market. Its primary GPU product line, labeled ")

print(f"{'TOKEN':<12}{'POS':<8}{'DEP':<12}{'HEAD'}")
for token in doc:
    print(f"{token.text:<12}{token.pos_:<8}{token.dep_:<12}{token.head.text}")

print("\nNamed entities:")
for ent in doc.ents:
    print(f"  {ent.text:<20} -> {ent.label_}")

TOKEN       POS     DEP         HEAD
Nvidia      PROPN   compound    Corporation
Corporation PROPN   nsubj       is
is          AUX     ROOT        is
an          DET     det         company
American    ADJ     amod        company
multinationalADJ     amod        technology
technology  NOUN    compound    company
company     NOUN    attr        is
incorporatedVERB    acl         company
in          ADP     prep        incorporated
Delaware    PROPN   pobj        in
and         CCONJ   cc          incorporated
based       VERB    conj        incorporated
in          ADP     prep        based
Santa       PROPN   compound    Clara
Clara       PROPN   pobj        in
,           PUNCT   punct       Clara
California  PROPN   appos       Clara
.           PUNCT   punct       is
It          PRON    nsubj       designs
designs     VERB    ROOT        designs
graphics    NOUN    compound    units
processing  NOUN    compound    units
units       NOUN    dobj        designs
(           PUNCT   pu

## PyThaiNLP example (Thai)

PyThaiNLP handles the Thai-specific problem we introduced in Section 1: **there are no spaces between Thai words**, so tokenization requires a dedicated algorithm/model rather than simply splitting on whitespace.
- https://pythainlp.org/

In [36]:
from pythainlp.tokenize import word_tokenize as th_word_tokenize
from pythainlp.tag import pos_tag as th_pos_tag
from pythainlp.corpus import thai_stopwords

thai_sentence = "อนุทิน ชาญวีรกูล นายกรัฐมนตรีของประเทศไทย เพื่อจะทำหให้ประชาชนพูดคำว่า รวยไม่ไหวแล้ว"

# 1. Word tokenization (default engine: "newmm", a dictionary-based maximal matching + TCC)
th_tokens = th_word_tokenize(thai_sentence)
print("Thai tokens:", th_tokens)

# 2. POS tagging
th_tagged = th_pos_tag(th_tokens)
print("\nThai POS tags:", th_tagged)

# 3. Stopword removal
th_stopwords = thai_stopwords()
th_stopwords = set(th_stopwords)  # Convert to a set for faster lookup
th_filtered = [w for w in th_tokens if w not in th_stopwords and w.strip() != ""]
print("\nThai tokens without stopwords:", th_filtered)

Thai tokens: ['อนุทิน', ' ', 'ชาญ', 'วีร', 'กู', 'ล', ' ', 'นายกรัฐมนตรี', 'ของ', 'ประเทศ', 'ไทย', ' ', 'เพื่อ', 'จะ', 'ทำ', 'ห', 'ให้', 'ประชาชน', 'พูด', 'คำ', 'ว่า', ' ', 'รวย', 'ไม่', 'ไหว', 'แล้ว']

Thai POS tags: [('อนุทิน', 'NCMN'), (' ', 'PUNC'), ('ชาญ', 'NCMN'), ('วีร', 'NCMN'), ('กู', 'NCMN'), ('ล', 'NCMN'), (' ', 'PUNC'), ('นายกรัฐมนตรี', 'NCMN'), ('ของ', 'RPRE'), ('ประเทศ', 'NCMN'), ('ไทย', 'NPRP'), (' ', 'PUNC'), ('เพื่อ', 'JSBR'), ('จะ', 'XVBM'), ('ทำ', 'VACT'), ('ห', 'NCMN'), ('ให้', 'JSBR'), ('ประชาชน', 'NCMN'), ('พูด', 'VACT'), ('คำ', 'NCMN'), ('ว่า', 'JSBR'), (' ', 'PUNC'), ('รวย', 'NCMN'), ('ไม่', 'NEG'), ('ไหว', 'VSTA'), ('แล้ว', 'XVAE')]

Thai tokens without stopwords: ['อนุทิน', 'ชาญ', 'วีร', 'ล', 'นายกรัฐมนตรี', 'ประเทศ', 'ไทย', 'ทำ', 'ห', 'ประชาชน', 'รวย', 'ไหว']


## Core Challenge Demo: Thai Word Segmentation Ambiguity

Let's revisit the ambiguous sentence from Section 1 and see how different tokenization **engines** inside PyThaiNLP can actually produce different segmentations — this is a live demonstration of why word segmentation is treated as its own hard NLP research problem for Thai, rather than a "solved" preprocessing step.

In [ ]:
from pythainlp.tokenize import word_tokenize

sentences = [
    "อาจารย์สอนภาษาไทยให้กับนักเรียนชาติจีน",
    "เขาตัดทรงผมใหม่แล้วดูดีมาก",
    
]

for sent in sentences:
    print(f"ประโยค: {sent}")
    for engine in ["newmm", "longest", "mm"]:
        try:
            result = word_tokenize(sent, engine=engine)
            print(f"  {engine:>10}: {result}")
        except Exception as e:
            print(f"  {engine:>10}: Error ({e})")
    print("-" * 40)


ประโยค: อาจารย์สอนภาษาไทยให้กับนักเรียนชาติจีน
       newmm: ['อาจารย์', 'สอน', 'ภาษาไทย', 'ให้', 'กับ', 'นักเรียน', 'ชาติ', 'จีน']
     longest: ['อาจารย์', 'สอน', 'ภาษาไทย', 'ให้', 'กับ', 'นักเรียน', 'ชาติ', 'จีน']
          mm: ['อาจารย์', 'สอนภาษาไทย', 'ให้', 'กับ', 'นักเรียน', 'ชาติ', 'จีน']
----------------------------------------
ประโยค: เขาตัดทรงผมใหม่แล้วดูดีมาก
       newmm: ['เขา', 'ตัด', 'ทรงผม', 'ใหม่', 'แล้ว', 'ดู', 'ดีมาก']
     longest: ['เขา', 'ตัด', 'ทรงผม', 'ใหม่', 'แล้ว', 'ดูดี', 'มาก']
          mm: ['เขา', 'ตัด', 'ทรงผม', 'ใหม่', 'แล้ว', 'ดูดีมาก']
----------------------------------------


## NLP Applications

### Text Classification

**Text classification** assigns a predefined label to a piece of text — e.g., sentiment (positive/negative), topic, spam/not-spam, or intent. It is a classic **NLU** task: text goes in, a structured label comes out.

Below is a minimal pipeline: turn text into numeric features (bag-of-words), then train a simple classifier. In real systems, the "text → numeric features" step is often replaced by upstream-learned embeddings (Word2Vec, BERT), but the classification idea stays the same.

In [25]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

# Toy training data: (review, label) where 1 = positive, 0 = negative
train_texts = [
    "I love this movie, it was fantastic",
    "What a wonderful and amazing film",
    "This was a terrible and boring movie",
    "I hated every minute of this film",
    "Absolutely brilliant, I really enjoyed it",
    "Awful acting, a complete waste of time",
]
train_labels = [1, 1, 0, 0, 1, 0]

vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(train_texts)

clf = LogisticRegression()
clf.fit(X_train, train_labels)

test_texts = ["This film was absolutely wonderful", "A boring and terrible experience", "i wow this game"]
X_test = vectorizer.transform(test_texts)
predictions = clf.predict(X_test)

for text, pred in zip(test_texts, predictions):
    print(f"{text!r:55} -> {'positive' if pred == 1 else 'negative'}")

'This film was absolutely wonderful'                    -> positive
'A boring and terrible experience'                      -> negative
'i wow this game'                                       -> negative


In [31]:
# Same idea, but for Thai text: we swap the tokenizer used by CountVectorizer
# for PyThaiNLP's word_tokenize, since Thai has no whitespace to split on.
from pythainlp.tokenize import word_tokenize as th_tokenize

th_train_texts = [
    "อาหารร้านนี้อร่อยมากแนะนำเลย",       # this restaurant's food is very delicious, recommended
    "บริการดีมากประทับใจสุดๆ",           # the service was great, very impressed
    "อาหารรสชาติแย่มากไม่อร่อยเลย",       # the food tasted very bad, not delicious at all
    "บริการห่วยแตกรอนานมากไม่ประทับใจ",   # terrible service, waited very long, not impressed
]
th_train_labels = [1, 1, 0, 0]

th_vectorizer = CountVectorizer(tokenizer=th_tokenize, token_pattern=None)
X_th_train = th_vectorizer.fit_transform(th_train_texts)

th_clf = LogisticRegression()
th_clf.fit(X_th_train, th_train_labels)

th_test_texts = ["ไม่มีใครเก่งเท่าแม่คุณแล้ว", "เก่งมากมั้ง"]
X_th_test = th_vectorizer.transform(th_test_texts)
th_predictions = th_clf.predict(X_th_test)

for text, pred in zip(th_test_texts, th_predictions):
    print(f"{text:35} -> {'positive' if pred == 1 else 'negative'}")

ไม่มีใครเก่งเท่าแม่คุณแล้ว          -> negative
เก่งมากมั้ง                         -> positive


## DIY Exercises

Now it's your turn. Each exercise has:

1. A **DIY** cell with `# TODO` markers for you to fill in.
2. An **Answer Key** cell right after it — try to solve the exercise first, then compare.

---

### Exercise 1 — Thai Tokenization & Counting

Using `pythainlp`, tokenize the sentence below and print:
1. The list of tokens.
2. The **number of unique tokens** (after removing Thai stopwords).

In [49]:
# === DIY Exercise 1 ===
from pythainlp.tokenize import word_tokenize
from pythainlp.corpus import thai_stopwords

sentence = "ข้าลูลูชบีบริทาเนียมหาราชขอบัญาให้พวกเจ้าทุกคนไปตายซะ"

# TODO 1: tokenize 'sentence' into a list of words using word_tokenize
tokens = sentence  # <-- replace None
tokens = word_tokenize(sentence)

# TODO 2: build a set of Thai stopwords using thai_stopwords()
stopwords_set = thai_stopwords()  # <-- replace None
stopwords_set = set(stopwords_set)  # Convert to a set for faster lookup

# TODO 3: build 'unique_content_tokens': the SET of tokens in 'tokens'
#         that are NOT in 'stopwords_set' and are not blank/whitespace
unique_content_tokens = {t for t in tokens if t not in stopwords_set and t.strip() != ""}  # <-- replace None
#HINT: unique_content_tokens = {t for t in ___ if t not in ___ and t.strip() != ""}

print("Tokens:", tokens)
print("Number of unique content tokens:", len(unique_content_tokens) if unique_content_tokens else "TODO not done yet")
print("Content tokens:", unique_content_tokens)

Tokens: ['ข้า', 'ลูลูช', 'บี', 'บริทาเนีย', 'มหาราช', 'ขอ', 'บัญา', 'ให้', 'พวก', 'เจ้า', 'ทุกคน', 'ไป', 'ตาย', 'ซะ']
Number of unique content tokens: 7
Content tokens: {'เจ้า', 'บัญา', 'มหาราช', 'ลูลูช', 'ตาย', 'บี', 'บริทาเนีย'}


---

### Exercise 2 — Simple English Stopword Filter + Word Frequency

Write a function `top_words(text, n)` that:
1. Tokenizes `text` with NLTK's `word_tokenize`.
2. Lowercases every token and keeps only alphabetic tokens (use `str.isalpha()`).
3. Removes English stopwords (`nltk.corpus.stopwords`).
4. Returns the `n` most common remaining words as a list of `(word, count)` tuples.

Hint: `collections.Counter` has a `.most_common(n)` method.

In [58]:
# === DIY Exercise 2 ===
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from collections import Counter

text = """
Natural language processing is a fascinating field. Natural language is complex,
ambiguous, and endlessly creative. Processing natural language well requires
understanding both language structure and meaning.
"""

def top_words(text, n):
    # TODO 1: tokenize 'text'
    tokens = text
    tokens = word_tokenize(text)
    print("Tokens:", tokens)
    # TODO 2: lowercase + keep only alphabetic tokens
    words = [t.lower() for t in tokens if t.isalpha()]
    print("Words:", words)
    #HINT: words = [t.lower() for t in ____ if _____ ]
    #HINT: text.isalpha() is use to check if a string contains only alphabetic characters

    # TODO 3: remove English stopwords
    stop_words = stopwords.words("english")
    filtered = [w for w in words if w not in stop_words and w.isalpha()]
    #HINT: stopwords.words("english") is used to get the list of English stopwords
    #HINT: filtered = [w for w in ____ if w not in _____]

    # TODO 4: count and return the n most common (word, count) pairs
    counts = Counter(filtered)
    print("Counts:", counts)
    #HINT: counts = Counter(_____)
    return counts.most_common(n)  # <-- return counts.most_common(____)

result = top_words(text, 5)
print(result)

Tokens: ['Natural', 'language', 'processing', 'is', 'a', 'fascinating', 'field', '.', 'Natural', 'language', 'is', 'complex', ',', 'ambiguous', ',', 'and', 'endlessly', 'creative', '.', 'Processing', 'natural', 'language', 'well', 'requires', 'understanding', 'both', 'language', 'structure', 'and', 'meaning', '.']
Words: ['natural', 'language', 'processing', 'is', 'a', 'fascinating', 'field', 'natural', 'language', 'is', 'complex', 'ambiguous', 'and', 'endlessly', 'creative', 'processing', 'natural', 'language', 'well', 'requires', 'understanding', 'both', 'language', 'structure', 'and', 'meaning']
Counts: Counter({'language': 4, 'natural': 3, 'processing': 2, 'fascinating': 1, 'field': 1, 'complex': 1, 'ambiguous': 1, 'endlessly': 1, 'creative': 1, 'well': 1, 'requires': 1, 'understanding': 1, 'structure': 1, 'meaning': 1})
[('language', 4), ('natural', 3), ('processing', 2), ('fascinating', 1), ('field', 1)]


---

### Exercise 3 — Named Entity Extraction with spaCy

Given the paragraph below, use spaCy to extract and print only the entities labeled **`PERSON`** and **`ORG`** (organization), in the format `"<text>  ->  <label>"`.

In [71]:
# === DIY Exercise 3 ===
import spacy
import os
import json
from spacy.tokens import DocBin
from spacy.util import filter_spans
import spacy.cli
nlp = spacy.load("en_core_web_sm")

paragraph = "Sundar Pichai, the CEO of Google, met with representatives from Microsoft"
doc = nlp(paragraph)

# TODO: loop over doc.ents and print only entities whose .label_ is
#       "PERSON" or "ORG", formatted as f"{ent.text}  ->  {ent.label_}"
result = [tok for tok in doc.ents if tok.label_ in ("PERSON", "ORG")]
for tok in result:
    print(f"{tok.text}  ->  {tok.label_}")
#HINT:
# for ____ in ___.ents:
#     if ____.label_ in ("PERSON", "ORG"):
#         print(f"{_____.text}  ->  {____.label_}")


Sundar Pichai  ->  PERSON
Google  ->  ORG
Microsoft  ->  ORG
